In [25]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [26]:
DATA_DIR =Path("data")
df = pd.read_csv(DATA_DIR / "product_info.csv") 
df.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


In [27]:
print("Number of products:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of products: 8494
Number of columns: 27


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   str    
 1   product_name        8494 non-null   str    
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   str    
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                6863 non-null   str    
 8   variation_type      7050 non-null   str    
 9   variation_value     6896 non-null   str    
 10  variation_desc      1250 non-null   str    
 11  ingredients         7549 non-null   str    
 12  price_usd           8494 non-null   float64
 13  value_price_usd     451 non-null    float64
 14  sale_price_usd      270 non-null    float64
 15  limited_edition     8494 non-null   int64  
 16  new              

In [29]:
df[["highlights", "price_usd", "ingredients", "size"]].isna().sum()

highlights     2207
price_usd         0
ingredients     945
size           1631
dtype: int64

In [30]:
# Products missing BOTH highlights and ingredients
both_missing = df[
    df["highlights"].isna() &
    df["ingredients"].isna()
]
print("Missing both:", len(both_missing))

# Products with both features available
complete = df[
    df["highlights"].notna() &
    df["ingredients"].notna()
]
print("Have both:", len(complete))

Missing both: 478
Have both: 5820


In [31]:
products = df.copy()

In [32]:
products = products.dropna(
    subset=["highlights", "ingredients"]
).copy()

products.reset_index(drop=True)

print(products.shape)

(5820, 27)


In [33]:
import ast

# convert str of list into python list
def parse_highlights(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return [str(value)]


products["highlights_list"] = products["highlights"].apply(
    parse_highlights
)

products[["highlights", "highlights_list"]].head()

,highlights,highlights_list
0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...","[Unisex/ Genderless Scent, Warm &Spicy Scent, ..."
1,"['Unisex/ Genderless Scent', 'Layerable Scent'...","[Unisex/ Genderless Scent, Layerable Scent, Wa..."
2,"['Unisex/ Genderless Scent', 'Layerable Scent'...","[Unisex/ Genderless Scent, Layerable Scent, Wo..."
3,"['Unisex/ Genderless Scent', 'Layerable Scent'...","[Unisex/ Genderless Scent, Layerable Scent, Wa..."
4,"['Unisex/ Genderless Scent', 'Layerable Scent'...","[Unisex/ Genderless Scent, Layerable Scent, Wo..."


In [34]:
from collections import Counter

highlight_counts = Counter(
    highlight.lower().strip()
    for highlights in products["highlights_list"]
    for highlight in highlights
)

highlight_df = pd.DataFrame(
    highlight_counts.most_common(),
    columns=["highlight", "frequency"]
)

highlight_df.head(50)

,highlight,frequency
0,vegan,2419
1,cruelty-free,1638
2,clean at sephora,1472
3,without parabens,1328
4,good for: dryness,1168
5,hydrating,1139
6,good for: dullness/uneven texture,884
7,without sulfates sls & sles,752
8,long-wearing,703
9,clean + planet positive,676


In [ ]:

import re
import unicodedata

def clean_highlight(text):
    # convert to lowercase
    text = str(text).lower().strip()

    # normalize unicode characters
    text = unicodedata.normalize("NFKC", text)

    # replace hyphens with spaces
    text = re.sub(r"[-‐‑–—]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text

products["cleaned_highlights"] = (
    products["highlights_list"].apply(
        lambda items: [clean_highlight(h) for h in items]
    )
)


In [36]:
unique_highlights = sorted({
    h.lower().strip()
    for highlights in products["cleaned_highlights"]
    for h in highlights
})

print(len(unique_highlights))
for h in unique_highlights:
    print(h)

112
aha/glycolic acid
alcohol free
all hair types
allure 2017 best of beauty award winner
allure 2018 best of beauty award winner
allure 2018 readers' choice award winner
allure 2019 best of beauty award winner
allure 2019: clean beauty winner
allure 2020 best of beauty award winner
allure 2021 best of beauty award winner
allure 2021: clean beauty winner
allure 2022 best of beauty award winner
allure 2022 clean beauty winner
allure readers' choice award winner: original foundation bareminerals
best for combination skin
best for dry skin
best for dry, combo, normal skin
best for normal skin
best for oily skin
best for oily, combo, normal skin
black owned at sephora
cbd
clean + planet positive
clean at sephora
coily hair
collagen
community favorite
cream formula
cruelty free
curl enhancing
curling
curly hair
customizable
floral scent
fragrance free
fresh scent
full coverage
gluten free
good for: acne/blemishes
good for: anti aging
good for: color care
good for: damage
good for: dark circ

In [37]:
#  now splite "good for: A,B,C" into "good for A, good for B ..."
skin_type_mapping = {
    "best for dry skin": ["skin_dry"],
    "best for oily skin": ["skin_oily"],
    "best for normal skin": ["skin_normal"],
    "best for combination skin": ["skin_combination"],
    "best for dry, combo, normal skin": [
        "skin_dry", "skin_combination", "skin_normal"
    ],
    "best for oily, combo, normal skin": [
        "skin_oily", "skin_combination", "skin_normal"
    ]
}

def normalize_skin_types(highlight):
    return skin_type_mapping.get(highlight, [highlight])



In [38]:
#  revomoving hihglights non-related
marketing_highlights = {
    "community favorite",
    "clean at sephora",
    "clean + planet positive",
    "black owned at sephora"
}

In [39]:
def normalize_highlight(highlight):
    # Normalize text formatting
    h = highlight.lower().strip()
    h = h.replace("-", " ")
    h = re.sub(r"\s+", " ", h)

    # Remove marketing features from similarity
    if h in marketing_highlights or h.startswith("allure "):
        return []

    # Expand combined skin-type labels
    if h in skin_type_mapping:
        return skin_type_mapping[h]

    # Normalize remaining highlights
    return [h.replace(" ", "_")]


    
def preprocess_highlights(highlights):
    normalized = []

    for highlight in highlights:
        normalized.extend(normalize_highlight(highlight))

    # Remove duplicates while preserving order
    return list(dict.fromkeys(normalized))


products["normalized_highlights"] = (
    products["cleaned_highlights"].apply(preprocess_highlights)
)


In [40]:
products[
    ["product_name", "highlights_list", "normalized_highlights"]
].head(10)

,product_name,highlights_list,normalized_highlights
0,Fragrance Discovery Set,"[Unisex/ Genderless Scent, Warm &Spicy Scent, ...","[unisex/_genderless_scent, warm_&spicy_scent, ..."
1,La Habana Eau de Parfum,"[Unisex/ Genderless Scent, Layerable Scent, Wa...","[unisex/_genderless_scent, layerable_scent, wa..."
2,Rainbow Bar Eau de Parfum,"[Unisex/ Genderless Scent, Layerable Scent, Wo...","[unisex/_genderless_scent, layerable_scent, wo..."
3,Kasbah Eau de Parfum,"[Unisex/ Genderless Scent, Layerable Scent, Wa...","[unisex/_genderless_scent, layerable_scent, wa..."
4,Purple Haze Eau de Parfum,"[Unisex/ Genderless Scent, Layerable Scent, Wo...","[unisex/_genderless_scent, layerable_scent, wo..."
5,Kasbah Eau de Parfum Travel Spray,"[Warm &Spicy Scent, Unisex/ Genderless Scent, ...","[warm_&spicy_scent, unisex/_genderless_scent, ..."
6,Purple Haze Eau de Parfum Travel Spray,"[Unisex/ Genderless Scent, Layerable Scent, Wo...","[unisex/_genderless_scent, layerable_scent, wo..."
7,Invisible Post Eau de Parfum,"[Unisex/ Genderless Scent, allure 2022 Best of...","[unisex/_genderless_scent, fresh_scent]"
8,Capri Eau de Parfum,"[Fresh Scent, Layerable Scent, Unisex/ Genderl...","[fresh_scent, layerable_scent, unisex/_genderl..."
9,Invisible Post Eau de Parfum Travel Spray,"[Unisex/ Genderless Scent, allure 2022 Best of...","[unisex/_genderless_scent, fresh_scent]"


In [41]:
unique_normalized = sorted({
    h
    for highlights in products["normalized_highlights"]
    for h in highlights
})

print("Unique normalized highlights:", len(unique_normalized))

for h in unique_normalized:
    print(h)

Unique normalized highlights: 95
aha/glycolic_acid
alcohol_free
all_hair_types
cbd
coily_hair
collagen
cream_formula
cruelty_free
curl_enhancing
curling
curly_hair
customizable
floral_scent
fragrance_free
fresh_scent
full_coverage
gluten_free
good_for:_acne/blemishes
good_for:_anti_aging
good_for:_color_care
good_for:_damage
good_for:_dark_circles
good_for:_dark_spots
good_for:_dryness
good_for:_dullness/uneven_texture
good_for:_flaky/dry_scalp
good_for:_frizz
good_for:_hair_thinning
good_for:_loss_of_firmness
good_for:_oily_scalp
good_for:_pores
good_for:_redness
good_for:_volume
heat_protection
high_shine_finish
hyaluronic_acid
hydrating
hypoallergenic
increases_shine
lactic_acid
layerable_scent
lengthening
light_coverage
light_hold
liquid_formula
long_wearing
loose_powder_formula
matte_finish
medium_coverage
medium_hold
metallic_finish
natural_finish
niacinamide
oil_free
plumping
pressed_powder_formula
prevents_color_fading
radiant_finish
recyclable_packaging
reduces_brassiness
reef

In [42]:
#  future consideration: I would also separate highlights into hard constraints and soft similarity features.

#  For example, if a customer specifically needs a fragrance-free moisturizer, a fragrance-containing moisturizer should not rank highly simply because its other characteristics are similar.

In [ ]:
import ast
import html
import re
import unicodedata
import pandas as pd


def normalize_ingredient(name):
    
    # clean some formatiing 
    if not isinstance(name, str):
        return ""

    #  decode some html values
    name = html.unescape(name)
    name = unicodedata.normalize("NFKC", name).casefold()

    # replace all possible dashes with one type
    name = re.sub(r"[‐‑‒–—−]", "-", name)

    # remove extra white spaces
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s*/\s*", "/", name)

    name = re.sub(r"[\s*†‡]+$", "", name)
    # removes speces, periods, semicolms
    return name.strip(" .;")

# "Water, Example Extract (leaf, stem), Glycerin" -> ["water", "example extract (leaf, stem)", "glycerin"]
def split_ingredient_text(text):
    #  to store the names of ingridients
    ingredients = []
    
    # the current chars of the ingridient
    current = []
    
    # opened parathethis 
    stack = []

    for char in text:
        if char in "([":
            stack.append(char)

        elif char in ")]":
            expected = "(" if char == ")" else "["
            if not stack or stack[-1] != expected:
                raise ValueError("unbalanced ingredient parentheses/brackets")
            stack.pop()

        if char == "," and not stack:
            ingredient = normalize_ingredient("".join(current))
            if ingredient:
                ingredients.append(ingredient)
            current = []
        else:
            current.append(char)

    if stack:
        raise ValueError("Unbalanced ingredient parentheses/brackets")

    ingredient = normalize_ingredient("".join(current))
    if ingredient:
        ingredients.append(ingredient)

    return ingredients


def parse_ingredients(value):
    if isinstance(value, list):
        sections = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return {"ingredients": [], "status": "missing"}

        if value.startswith("["):
            try:
                # converts text into actual list python
                sections = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return {"ingredients": [], "status": "invalid_list"}
        else:
            sections = [value]

    else:
        return {"ingredients": [], "status": "missing"}

    if not isinstance(sections, list):
        return {"ingredients": [], "status": "invalid_list"}

    if not all(isinstance(section, str) for section in sections):
        return {"ingredients": [], "status": "invalid_section"}

    sections = [section.strip() for section in sections if section.strip()]

    if not sections:
        return {"ingredients": [], "status": "missing"}

    if len(sections) > 1:
        return {"ingredients": [], "status": "needs_section_review"}

    try:
        ingredients = split_ingredient_text(sections[0])
    except ValueError:
        return {"ingredients": [], "status": "needs_delimiter_review"}

    return {
        "ingredients": ingredients,
        "status": "parsed" if ingredients else "empty",
    }

In [44]:
for i in range(5):
    print(f"\nPRODUCT: {products.iloc[i]['product_name']}")
    print("RAW INGREDIENTS:")
    print(repr(products.iloc[i]["ingredients"]))
    print("-" * 100)


PRODUCT: Fragrance Discovery Set
RAW INGREDIENTS:
"['Capri Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) D-Limonene, Linalool, Benzyl Salicylate, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, Benzl Benzoate, Citral, Geraniol, Eugenol, Benzyl Alcohol, Farnesol, Citronellol, Isoeugenol.', 'Invisible Post Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Ethylhexyl Methoxycinnamate, Ethylhexyl Salicylate, Butyl Methoxydibenzoylmethane, Benzyl Benzoate, Citral, Coumarin, Limonene, Alpha-Isomethyl Ionone, Linalool.', 'Kashbah Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Coumarin, Ethylhexyl Methoxycinnamate, Butyl Methoxydibenzoylmethane, Ethylhexyl Salicylate, D-Limonene, Eugenol, Linalool, Citronellol, Geraniol, Cinnamal, Citral.', 'L’Air Barbes Eau de Parfum:', 'Alcohol Denat. (SD Alcohol 39C), Parfum (Fragrance) Benzyl Salicylate, D-Limonene, Ethylhexyl Methoxycinnamate, Butyl Meth

In [ ]:
APPROVED_CORRECTIONS = {
    "hydroxyacetophe-none": "hydroxyacetophenone",
    "pan-thenol": "panthenol",
}

APPROVED_ALIASES = {
    "aqua/water/eau": "water",
    "water/aqua/eau": "water",
    "aqua (water)": "water",
    "water (aqua)": "water",
}

CANONICAL_MAP = {
    normalize_ingredient(source): normalize_ingredient(target)
    for source, target in {
        **APPROVED_CORRECTIONS,
        **APPROVED_ALIASES,
    }.items()
}


def canonicalize_ingredients(ingredients):
    # applying the approved replacement
    cleaned = []

    for ingredient in ingredients:
        name = normalize_ingredient(ingredient)
        name = CANONICAL_MAP.get(name, name)

        if name:
            cleaned.append(name)

    return list(dict.fromkeys(cleaned))




In [ ]:
parsed = pd.DataFrame(
    products["ingredients"].apply(parse_ingredients).tolist(),
    index=products.index,
)

products["ingredients_list"] = parsed["ingredients"]
products["ingredient_parse_status"] = parsed["status"]


products["normalized_ingredients"] = (
    products["ingredients_list"].apply(canonicalize_ingredients)
)

print(products["ingredient_parse_status"].value_counts())

# recreate skincare after updating products.
skincare = products.loc[
    products["primary_category"].eq("Skincare")
].copy()

<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.


ingredient_parse_status
parsed                    4987
needs_section_review       798
needs_delimiter_review      35
Name: count, dtype: int64


In [ ]:
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein


def extract_numbers(name):
    return tuple(re.findall(r"\d+(?:\.\d+)?", name))


def suggest_corrections(
    ingredient_names,
    reference_names,
    max_distance=0.15,
    limit=3,
):
    reference = sorted({
        normalize_ingredient(name)
        for name in reference_names
        if isinstance(name, str) and normalize_ingredient(name)
    })
    reference_set = set(reference)
    rows = []

    for name in sorted({
        normalize_ingredient(name)
        for name in ingredient_names
        if isinstance(name, str) and normalize_ingredient(name)
    }):
        if name in reference_set:
            continue

        # matching numbers is a filter not proof of equivalence.
        eligible = [
            candidate
            for candidate in reference
            if extract_numbers(candidate) == extract_numbers(name)
        ]

        matches = process.extract(
            name,
            eligible,
            scorer=Levenshtein.normalized_distance,
            score_cutoff=max_distance,
            limit=limit,
        )

        for candidate, distance, _ in matches:
            rows.append({
                "ingredient": name,
                "suggested_name": candidate,
                "distance": distance,
                "approved": False,
            })

    return pd.DataFrame(
        rows,
        columns=[
            "ingredient",
            "suggested_name",
            "distance",
            "approved",
        ],
    )

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

skincare_for_similarity = skincare.loc[
    skincare["ingredient_parse_status"].eq("parsed")
    & skincare["normalized_ingredients"].map(bool)
].copy()

vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

X_ingredients = vectorizer.fit_transform(
    skincare_for_similarity["normalized_ingredients"]
)

In [51]:
skincare = products.loc[
    products["primary_category"].eq("Skincare")
].copy()

print(skincare["ingredient_parse_status"].value_counts())

skincare_model = skincare.loc[
    skincare["ingredient_parse_status"].eq("parsed")
    & skincare["normalized_ingredients"].map(bool)
].copy()

print("Skincare products available for modeling:", len(skincare_model))

ingredient_parse_status
parsed                    1680
needs_section_review       245
needs_delimiter_review       2
Name: count, dtype: int64
Skincare products available for modeling: 1680


In [67]:
vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

X_ingredients = vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

In [68]:
vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

X_ingredients = vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

ingredient_similarity = cosine_similarity(X_ingredients)

print("TF-IDF shape:", X_ingredients.shape)
print("Similarity matrix shape:", ingredient_similarity.shape)

TF-IDF shape: (1525, 5156)
Similarity matrix shape: (1525, 1525)


In [69]:
# load the ingridient all possible from the EU ingridirct dictionary 

inci_dictionary = pd.read_csv("data/inci_dictionary.csv")

reference_names = (
    inci_dictionary["inci_name"]
    .map(normalize_ingredient)
    .drop_duplicates()
    .tolist()
)

print("Reference names:", len(reference_names))
display(inci_dictionary.head())

Reference names: 30412


,entry_number,inci_name
0,1,(20S)-PROTOPANAXADIOL
1,2,(ACORI GRAMINEI/DISCOREA JAPONICA/PUERARIA MON...
2,3,(ACTAEA HERACLEIFOLIA/PAEONIA LATIFLORA/RHEUM ...
3,4,(ADONIS AMURENSIS/AKEBIA QUINATA/ALBIZIA JULIB...
4,5,(AGARICUS BLAZEI FERMENT + GANODERMA LUCIDUM F...


In [70]:
ingredient_names = {
    ingredient
    for ingredients in skincare_for_similarity["normalized_ingredients"]
    for ingredient in ingredients
}

suggestions = suggest_corrections(
    ingredient_names=ingredient_names,
    reference_names=reference_names,
    max_distance=0.10,
    limit=3,
)

print("Suggested corrections:", len(suggestions))
display(suggestions.head(20))

Suggested corrections: 280


,ingredient,suggested_name,distance,approved
0,*batyl alcohol,batyl alcohol,0.071429,False
1,*behenyl alcohol,behenyl alcohol,0.062500,False
2,3-oethyl ascorbic acid,3-o-ethyl ascorbic acid,0.043478,False
3,4-t- butylcyclohexanol,4-t-butylcyclohexanol,0.045455,False
4,acetyl hexapeptide-51 (amide),acetyl hexapeptide-51 amide,0.068966,False
5,acetylcysteine,acetyl cysteine,0.066667,False
6,acetylheptapeptide-4,acetyl heptapeptide-4,0.047619,False
7,acrylate/sodium acryloyldimenthyl taurate copo...,acrylamide/sodium acryloyldimethyltaurate copo...,0.098039,False
8,acrylates/c10 -30 alkyl acrylate crosspolymer,acrylates/c10-30 alkyl acrylate crosspolymer,0.022222,False
9,acrylates/stearyl methacrylate opolymer,acrylates/stearyl methacrylate copolymer,0.025000,False


In [72]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Keep individual skincare formulas.
eligible_categories = {
    "Cleansers",
    "Eye Care",
    "Lip Balms & Treatments",
    "Masks",
    "Moisturizers",
    "Self Tanners",
    "Sunscreen",
    "Treatments",
}

skincare_model = products.loc[
    products["primary_category"].eq("Skincare")
    & products["secondary_category"].isin(eligible_categories)
    & products["ingredient_parse_status"].eq("parsed")
    & products["normalized_ingredients"].map(bool)
].copy()

skincare_model["price_usd"] = pd.to_numeric(
    skincare_model["price_usd"],
    errors="coerce",
)

skincare_model = (
    skincare_model
    .drop_duplicates(subset="product_id")
    .reset_index(drop=True)
)

if skincare_model.empty:
    raise ValueError("No eligible products. Check the preprocessing cells.")

print("Products available:", len(skincare_model))

Products available: 1525


In [73]:
from sklearn.preprocessing import MultiLabelBinarizer

# Keep dataframe rows aligned with both matrices.
skincare_model = skincare_model.reset_index(drop=True)

ingredient_vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

X_ingredients = ingredient_vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

# Each highlight becomes a binary feature: present or absent.
highlight_encoder = MultiLabelBinarizer(sparse_output=True)

X_highlights = highlight_encoder.fit_transform(
    skincare_model["normalized_highlights"]
)

product_positions = {
    product_id: position
    for position, product_id in enumerate(skincare_model["product_id"])
}

In [74]:
ingredient_vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

X_ingredients = ingredient_vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

# Sets are useful for ingredient overlap and the Jaccard baseline.
ingredient_sets = [
    set(ingredients)
    for ingredients in skincare_model["normalized_ingredients"]
]

# Find a product's matrix position from its ID.
product_positions = {
    product_id: position
    for position, product_id in enumerate(skincare_model["product_id"])
}

print("TF-IDF matrix shape:", X_ingredients.shape)

TF-IDF matrix shape: (1525, 5156)


In [75]:
def recommend_dupes(
    product_id,
    top_k=5,
    ingredient_weight=0.7,
    different_brand=False,
):
    if product_id not in product_positions:
        raise ValueError("Product ID is not in skincare_model.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k must be a positive integer.")

    if not 0 < ingredient_weight < 1:
        raise ValueError("ingredient_weight must be between 0 and 1.")

    position = product_positions[product_id]
    query = skincare_model.iloc[position]

    if pd.isna(query["price_usd"]) or query["price_usd"] <= 0:
        raise ValueError("The selected product needs a valid price.")

    if X_highlights[position].nnz == 0:
        raise ValueError(
            "This product has no usable highlights to compare."
        )

    # Calculate the two similarity scores separately.
    ingredient_scores = cosine_similarity(
        X_ingredients[position],
        X_ingredients,
    ).ravel()

    highlight_scores = cosine_similarity(
        X_highlights[position],
        X_highlights,
    ).ravel()

    combined_scores = (
        ingredient_weight * ingredient_scores
        + (1 - ingredient_weight) * highlight_scores
    )

    # Only consider comparable product categories.
    mask = skincare_model["secondary_category"].eq(
        query["secondary_category"]
    )

    if pd.notna(query["tertiary_category"]):
        mask &= skincare_model["tertiary_category"].eq(
            query["tertiary_category"]
        )
    else:
        mask &= skincare_model["tertiary_category"].isna()

    # Alternatives MUST have a lower package price.
    mask &= (
        skincare_model["price_usd"].gt(0)
        & skincare_model["price_usd"].lt(query["price_usd"])
    )

    mask &= skincare_model["product_id"].ne(product_id)

    if different_brand:
        mask &= skincare_model["brand_name"].ne(query["brand_name"])

    # Require some overlap in BOTH ingredients and highlights.
    mask &= (ingredient_scores > 0) & (highlight_scores > 0)

    results = skincare_model.loc[mask].copy()
    selected = mask.to_numpy()

    results["ingredient_similarity"] = ingredient_scores[selected]
    results["highlight_similarity"] = highlight_scores[selected]
    results["combined_similarity"] = combined_scores[selected]

    results = results.sort_values(
        ["combined_similarity", "price_usd", "product_id"],
        ascending=[False, True, True],
    ).head(top_k).copy()

    # Explain what each alternative shares with the selected product.
    query_ingredients = set(query["normalized_ingredients"])
    query_highlights = set(query["normalized_highlights"])

    results["shared_ingredients"] = results[
        "normalized_ingredients"
    ].apply(lambda items: sorted(query_ingredients & set(items)))

    results["shared_highlights"] = results[
        "normalized_highlights"
    ].apply(lambda items: sorted(query_highlights & set(items)))

    results["package_savings_usd"] = (
        query["price_usd"] - results["price_usd"]
    )

    results["package_savings_pct"] = (
        100 * results["package_savings_usd"] / query["price_usd"]
    )

    columns = [
        "product_id",
        "product_name",
        "brand_name",
        "size",
        "price_usd",
        "ingredient_similarity",
        "highlight_similarity",
        "combined_similarity",
        "package_savings_usd",
        "package_savings_pct",
        "shared_highlights",
        "shared_ingredients",
    ]

    return results[columns].reset_index(drop=True)

In [77]:
query_id = skincare_model.iloc[0]["product_id"]

display(
    skincare_model.loc[
        skincare_model["product_id"].eq(query_id),
        [
            "product_name",
            "price_usd",
            "size",
            "normalized_highlights",
        ],
    ]
)

recommendations = recommend_dupes(
    product_id=query_id,
    top_k=5,
    ingredient_weight=0.7,
)

display(recommendations.round(3))

if recommendations.empty:
    print("No cheaper alternatives match both ingredients and highlights.")

,product_name,price_usd,size,normalized_highlights
0,GENIUS Sleeping Collagen Moisturizer,98.0,2 oz/ 60 mL,"[vegan, good_for:_loss_of_firmness, collagen, ..."


,product_id,product_name,brand_name,size,price_usd,ingredient_similarity,highlight_similarity,combined_similarity,package_savings_usd,package_savings_pct,shared_highlights,shared_ingredients
0,P453818,GENIUS Collagen Calming Relief,Algenist,1.35 oz/ 40 mL,58.00,0.214,0.802,0.391,40.00,40.816,"[collagen, hypoallergenic, skin_combination, s...","[caprylyl glycol, cetearyl alcohol, chlorella ..."
1,P411403,Confidence in a Cream Anti-Aging Hydrating Moi...,IT Cosmetics,0.51 oz / 15 mL,20.00,0.105,0.750,0.298,78.00,79.592,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, caprylyl glycol..."
2,P427641,Mini Confidence in a Cream Hydrating Moisturizer,IT Cosmetics,0.51 oz / 15 mL,20.00,0.105,0.750,0.298,78.00,79.592,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, caprylyl glycol..."
3,P405599,SEA drink of H2O hydrating boost moisturizer,tarte,1.70 oz/ 50 mL,39.00,0.092,0.625,0.252,59.00,60.204,"[good_for:_loss_of_firmness, skin_combination,...",[ammonium acryloyldimethyltaurate/vp copolymer...
4,P455369,Peptide Moisturizer,The INKEY List,1.7 oz/ 50 mL,15.99,0.104,0.577,0.246,82.01,83.684,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, cetearyl alcoho..."


In [65]:
jaccard_recommendations = recommend_dupes(
    product_id=query_id,
    top_k=5,
    cheaper_only=True,
    method="jaccard",
)

display(jaccard_recommendations)


,product_id,product_name,brand_name,secondary_category,tertiary_category,size,price_usd,similarity,package_savings_usd,package_savings_pct,shared_count,shared_ingredients
0,P469862,Rich Bitch Cactus + Vitamin C Moisturizer,Freck Beauty,Moisturizers,Moisturizers,1.7 oz/ 50 mL,34.0,0.216667,64.0,65.306122,13,"[caprylic/capric triglyceride, caprylyl glycol..."
1,P483698,Nourishing Squalane Daily Moisturizer,The Outset,Moisturizers,Moisturizers,1.7 oz / 50 mL,44.0,0.208333,54.0,55.102041,15,"[caprylic/capric triglyceride, caprylyl glycol..."
2,P453818,GENIUS Collagen Calming Relief,Algenist,Moisturizers,Moisturizers,1.35 oz/ 40 mL,58.0,0.189189,40.0,40.816327,14,"[caprylyl glycol, cetearyl alcohol, chlorella ..."
3,P501282,C5 Super Boost Vitamin C Moisturizer,Paula's Choice,Moisturizers,Moisturizers,1.7 oz / 50 mL,45.0,0.186667,53.0,54.081633,14,"[caprylyl glycol, citric acid, dimethicone, et..."
4,P481701,Cloud Cushion Plush Moisturizer with Ceramides...,EADEM,Moisturizers,Moisturizers,1.69 oz/ 50 mL,58.0,0.185714,40.0,40.816327,13,"[caprylic/capric triglyceride, caprylyl glycol..."
